# Fine-Tuning LLaMA 3.1-8B for Solana Vulnerability Detection

This notebook trains LLaMA 3.1-8B-Instruct using QLoRA to detect security vulnerabilities in Solana smart contracts written in Rust.

The training approach follows the Chain-of-Thought (CoT) methodology, where the model learns to reason step-by-step before issuing a security verdict. This replicates and adapts the pipeline proposed by Tortora (2025) for Algorand, applying it to the Solana ecosystem.

**Reference paper:** Boi, B. & Esposito, C. (2025). Prompt Engineering vs. Fine-Tuning for LLM-Based Vulnerability Detection in Solana and Algorand Smart Contracts. IEEE BCCA 2025.

**Hardware requirement:** Kaggle T4 GPU (16GB VRAM)

## 1. Environment Setup

Install Unsloth (optimized fine-tuning framework) and its dependencies. Unsloth reduces VRAM usage by approximately 30%, which is critical for fitting an 8B model on a T4 GPU.

In [ ]:
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. Data Verification

Before training, we verify that the uploaded dataset files are accessible. These files should be added via the "Add Input" panel on the right side of Kaggle, from the dataset named `solana-training-data`.

In [ ]:
import os
import json

TRAIN_PATH = "/kaggle/input/datasets/mustafahafed/solana-training-data/dataset_think_format.jsonl"
VAL_PATH = "/kaggle/input/datasets/mustafahafed/solana-training-data/validation_dataset.jsonl"

for path, name in [(TRAIN_PATH, "Training"), (VAL_PATH, "Validation")]:
    if os.path.exists(path):
        with open(path) as f:
            count = sum(1 for _ in f)
        print(f"{name} data: {count} samples ({path})")
    else:
        raise FileNotFoundError(f"{name} data not found at {path}")

## 3. Data Preparation

The raw dataset stores the Chain-of-Thought reasoning and the final verdict in separate fields (`thinking` and `content`). For training with LLaMA's native chat template, we merge them into a single assistant response wrapped in `<think>` and `<final>` tags.

This step also converts the `developer` role to `system`, which is the standard role name expected by LLaMA's tokenizer.

In [ ]:
def prepare_dataset(input_path, output_path):
    """Merge the separate thinking/content fields into a single CoT response."""
    prepared = []
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            sample = json.loads(line)
            new_messages = []

            for msg in sample["messages"]:
                if msg["role"] == "assistant" and "thinking" in msg:
                    merged = (
                        f"<think>\n{msg['thinking']}\n</think>\n\n"
                        f"<final>\n{msg['content']}\n</final>"
                    )
                    new_messages.append({"role": "assistant", "content": merged})
                elif msg["role"] == "developer":
                    new_messages.append({"role": "system", "content": msg["content"]})
                else:
                    new_messages.append(msg)

            prepared.append({"id": sample["id"], "messages": new_messages})

    with open(output_path, "w", encoding="utf-8") as f:
        for item in prepared:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    return len(prepared)

os.makedirs("/kaggle/working/prepared", exist_ok=True)

n_train = prepare_dataset(TRAIN_PATH, "/kaggle/working/prepared/train.jsonl")
n_val = prepare_dataset(VAL_PATH, "/kaggle/working/prepared/val.jsonl")

print(f"Training samples prepared: {n_train}")
print(f"Validation samples prepared: {n_val}")

# Quick sanity check on the merged format
with open("/kaggle/working/prepared/train.jsonl") as f:
    sample = json.loads(f.readline())
    response = sample["messages"][-1]["content"]
    assert "<think>" in response and "<final>" in response, "Format error"
    print(f"\nSample response preview:\n{response[:300]}...")

## 4. Load Base Model

We load Meta's LLaMA 3.1-8B-Instruct with 4-bit quantization (NF4). This reduces the memory footprint from approximately 16GB to 6GB, leaving enough room on the T4 for training gradients and optimizer states.

The Unsloth distribution of the model is pre-optimized for efficient fine-tuning.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    full_finetuning=False,
)

print("Base model loaded: LLaMA 3.1-8B-Instruct (4-bit)")

## 5. LoRA Adapter Configuration

Instead of updating all 8 billion parameters (which would require hundreds of GB of memory), we use Low-Rank Adaptation (LoRA). This technique freezes the original weights and injects small trainable matrices into the model's attention and feed-forward layers.

The configuration follows Tortora (2025), Section 4.4.1:
- Rank r = 64 (higher rank captures more complex patterns)
- Alpha = 128 (scaling factor, typically 2x the rank)
- Target: all linear layers (attention + MLP), not just attention heads

With these settings, only about 2% of the total parameters are trainable.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 6. Dataset Loading and Tokenization

We load the prepared JSONL files and apply LLaMA's chat template. The template converts the structured messages (system, user, assistant) into the token format that LLaMA expects during training.

The `standardize_sharegpt` function ensures consistent field naming across different dataset conventions.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only

def formatting_prompts_func(examples):
    """Apply the LLaMA chat template to each conversation."""
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

dataset_train = load_dataset(
    "json", data_files="/kaggle/working/prepared/train.jsonl", split="train"
).shuffle(seed=43)

dataset_eval = load_dataset(
    "json", data_files="/kaggle/working/prepared/val.jsonl", split="train"
).shuffle(seed=43)

dataset_train = standardize_sharegpt(dataset_train)
dataset_train = dataset_train.map(formatting_prompts_func, batched=True)

dataset_eval = standardize_sharegpt(dataset_eval)
dataset_eval = dataset_eval.map(formatting_prompts_func, batched=True)

print(f"Training samples: {len(dataset_train)}")
print(f"Validation samples: {len(dataset_eval)}")

## 7. Training Configuration

All hyperparameters follow Tortora (2025), Table 4.1:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Epochs | 3 | Sufficient for convergence without overfitting on a small dataset |
| Effective batch size | 8 | batch_size=1 with 8 gradient accumulation steps |
| Learning rate | 5e-5 | Standard for LoRA fine-tuning |
| Scheduler | Cosine with 5% warmup | Gradual warmup prevents early instability |
| Optimizer | Paged AdamW 8-bit | Memory-efficient variant for GPU constraints |

The `train_on_responses_only` strategy ensures that the loss is computed only on the assistant's output (the CoT reasoning and verdict), not on the system prompt or the input code. This focuses the learning signal on what matters: the analysis quality.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,

    num_train_epochs=3,
    max_seq_length=MAX_SEQ_LENGTH,

    learning_rate=5e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    weight_decay=0.02,
    optim="paged_adamw_8bit",
    max_grad_norm=1.0,

    eval_strategy="steps",
    eval_steps=10,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=3,

    seed=3407,
    output_dir="/kaggle/working/models/LLaMA-3.1-8B-Solana-Audit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train,
    eval_dataset=dataset_eval,
    args=sft_config,
)

# Mask the loss on system and user tokens so the model only
# learns from the assistant response (CoT + verdict)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>",
    response_part="<|start_header_id|>assistant<|end_header_id|>",
)

print("Trainer configured. Ready to start.")

## 8. Training

This cell runs the actual fine-tuning. On a Kaggle T4, expect approximately 30-60 minutes for 3 epochs over 226 samples.

During training, monitor two values:
- **train_loss**: should decrease steadily (the model is learning)
- **eval_loss**: should decrease then stabilize (generalization check)

If eval_loss starts increasing while train_loss keeps dropping, the model is memorizing instead of learning. With 3 epochs this is unlikely, but the validation set provides an early warning.

In [ ]:
trainer_stats = trainer.train()

print(f"\nTraining completed in {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Final training loss: {trainer_stats.metrics['train_loss']:.4f}")

## 9. Save the Trained Adapter

We save only the LoRA adapter weights, not the full 8B model. The adapter is typically 100-200MB compared to the 16GB base model. To use the fine-tuned model later, we load the base model and merge the adapter on top.

In [ ]:
SAVE_DIR = "/kaggle/working/models/LLaMA-3.1-8B-Solana-Audit"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

total_size = sum(
    os.path.getsize(os.path.join(SAVE_DIR, f))
    for f in os.listdir(SAVE_DIR)
) / (1024 * 1024)

print(f"Adapter saved to {SAVE_DIR}")
print(f"Total size: {total_size:.1f} MB")

## 10. Inference Test

A quick sanity check to verify the fine-tuned model produces structured output with the expected `<think>` and `<final>` tags. We test it on a simple Solana function with a deliberate vulnerability (missing signer verification).

In [ ]:
FastLanguageModel.for_inference(model)

test_code = """
pub fn process_withdraw(
    program_id: &Pubkey,
    accounts: &[AccountInfo],
    amount: u64,
) -> ProgramResult {
    let authority_info = next_account_info(account_info_iter)?;
    let vault_info = next_account_info(account_info_iter)?;
    let destination_info = next_account_info(account_info_iter)?;

    let transfer_ix = spl_token::instruction::transfer(
        token_program.key, vault_info.key,
        destination_info.key, authority_info.key,
        &[], amount,
    )?;
    invoke(&transfer_ix, &[vault_info.clone(), destination_info.clone(), authority_info.clone()])?;
    Ok(())
}
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are an expert smart contract security auditor specialized in "
            "the Solana blockchain and Rust. Analyze the code for vulnerabilities. "
            "Put your reasoning inside <think> tags and your verdict inside <final> tags."
        ),
    },
    {
        "role": "user",
        "content": f"Analyze this Solana contract for vulnerabilities:\n\n{test_code}",
    },
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs, max_new_tokens=512, temperature=0.6, top_p=0.95
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("Model response:")
print("=" * 60)
print(response)
print("=" * 60)
print(f"\nContains <think>: {'<think>' in response}")
print(f"Contains <final>: {'<final>' in response}")

## 11. Export

Compress the adapter files for download. This archive will be needed in the evaluation phase to test the fine-tuned model against the test set.

In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/LLaMA-3.1-8B-Solana-Audit",
    "zip",
    SAVE_DIR,
)

size_mb = os.path.getsize(archive_path) / (1024 * 1024)
print(f"Exported: {archive_path} ({size_mb:.1f} MB)")
print("Download this file from the Output tab on the right panel.")

In [ ]:
model.push_to_hub_merged(
    "Mustafa99Hafed/LLaMA-3.1-8B-Solana-Audit",
    tokenizer,
    save_method="lora",
    token="YOUR_HF_TOKEN_HERE",
)
print("Done!")